# Online Retail II — Data Exploration

Companion notebook for **MarginBoard**. Goal: build a defensible mental model of the dataset before the dashboard and ML services consume it.

Specifically:
1. Confirm column meanings and types.
2. Quantify missingness, returns, and price/quantity outliers.
3. Check temporal coverage and seasonality.
4. Describe country and SKU concentration.
5. Set defensible thresholds for the inventory and anomaly modules.

If you change the cleaning rules here, mirror them in `backend/app/services/data_service.py`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

In [ ]:
RAW_PATH = Path("..") / "backend" / "data" / "raw" / "online_retail_II.csv"
assert RAW_PATH.exists(), f"Place the source CSV at {RAW_PATH}"

raw = pd.read_csv(RAW_PATH, low_memory=False)
raw.shape

## 1. Columns and types

The dataset uses `Invoice` / `Price` / `Customer ID` rather than `InvoiceNo` / `UnitPrice` / `CustomerID`. The data loader handles both.

In [ ]:
raw.dtypes

In [ ]:
raw.head()

## 2. Missingness

`Customer ID` is missing for a meaningful share of rows. Those rows still represent real revenue and must not be dropped — only the active-customer metric should ignore them.

In [ ]:
missing = raw.isna().mean().mul(100).round(2).sort_values(ascending=False)
missing.to_frame("missing_pct")

## 3. Cleaning preview

Same rules as `data_service.py`: standardize columns, coerce types, drop invalid prices, keep negative quantities (return signal).

In [ ]:
df = raw.rename(columns={
    "Invoice": "invoice_id",
    "StockCode": "stock_code",
    "Description": "description",
    "Quantity": "quantity",
    "InvoiceDate": "invoice_date",
    "Price": "unit_price",
    "Customer ID": "customer_id",
    "Country": "country",
})
df["invoice_date"] = pd.to_datetime(df["invoice_date"], errors="coerce")
df = df.dropna(subset=["invoice_id", "stock_code", "invoice_date", "quantity", "unit_price"])
df = df[df["unit_price"] >= 0]
df["revenue"] = df["quantity"] * df["unit_price"]
print(f"Rows after cleaning: {len(df):,}")
df.describe(percentiles=[0.5, 0.9, 0.99]).round(2)

## 4. Returns and cancellations

Invoices prefixed `C` are cancellations. Negative quantity rows are returns. We keep them because they affect net revenue.

In [ ]:
cancellations = df[df["invoice_id"].str.startswith("C", na=False)]
neg_qty = df[df["quantity"] < 0]
print(f"Cancellation invoices : {cancellations['invoice_id'].nunique():,} ({len(cancellations):,} lines)")
print(f"Negative quantity rows: {len(neg_qty):,}")
print(f"Return revenue impact : {neg_qty['revenue'].sum():,.2f}")

## 5. Temporal coverage

Daily revenue series powers the forecast. Reindex to a continuous range and inspect zero-revenue days — they break naive lag features if not handled.

In [ ]:
print(f"First invoice: {df['invoice_date'].min()}")
print(f"Last invoice : {df['invoice_date'].max()}")

daily = (
    df.assign(day=df["invoice_date"].dt.normalize())
      .groupby("day")["revenue"].sum()
)
full = pd.date_range(daily.index.min(), daily.index.max(), freq="D")
daily = daily.reindex(full, fill_value=0.0)

zero_days = (daily == 0).sum()
print(f"Zero-revenue days: {zero_days} of {len(daily)} ({zero_days/len(daily):.1%})")

In [ ]:
fig, ax = plt.subplots()
daily.plot(ax=ax, linewidth=0.8, color="#2563EB")
ax.set_title("Daily revenue")
ax.set_ylabel("GBP")
ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Day-of-week effect

In [ ]:
dow = daily.groupby(daily.index.day_name()).mean()
order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
dow = dow.reindex(order)

fig, ax = plt.subplots(figsize=(8, 3))
dow.plot(kind="bar", ax=ax, color="#2563EB")
ax.set_title("Average daily revenue by day of week")
ax.set_ylabel("GBP")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout(); plt.show()

## 6. Country concentration

Revenue is dominated by one or two markets. This matters for the dashboard's country filter — most of the variance lives in a handful of bars.

In [ ]:
country_rev = df.groupby("country")["revenue"].sum().sort_values(ascending=False)
share = country_rev.div(country_rev.sum()).mul(100).round(2)
share.head(10).to_frame("share_pct")

## 7. SKU concentration

Long tail: a small set of SKUs drives most of the revenue. The top-products table on the Overview page is meaningful — most of the catalog is noise.

In [ ]:
sales = df[df["quantity"] > 0]
sku_rev = sales.groupby("stock_code")["revenue"].sum().sort_values(ascending=False)
cumshare = sku_rev.cumsum() / sku_rev.sum()
k_for_80 = int((cumshare < 0.8).sum()) + 1
print(f"Total SKUs: {len(sku_rev):,}")
print(f"SKUs needed to cover 80% of revenue: {k_for_80:,} ({k_for_80/len(sku_rev):.1%})")

## 8. Outlier thresholds used downstream

The anomaly service uses the 99th percentile of absolute quantity, unit price, and absolute transaction value as rule thresholds. These are the values it will use on this dataset:

In [ ]:
abs_qty = df["quantity"].abs()
abs_val = (df["quantity"] * df["unit_price"]).abs()
pd.DataFrame({
    "feature": ["|quantity|", "unit_price", "|transaction_value|"],
    "p50": [abs_qty.quantile(0.5), df["unit_price"].quantile(0.5), abs_val.quantile(0.5)],
    "p99": [abs_qty.quantile(0.99), df["unit_price"].quantile(0.99), abs_val.quantile(0.99)],
    "max": [abs_qty.max(), df["unit_price"].max(), abs_val.max()],
}).round(2)

## 9. Takeaways for the dashboard

- Two years of UK-heavy retail data; treat global comparisons with care.
- Missing `customer_id` for a non-trivial share of rows — keep them in revenue but exclude from active-customer counts.
- Strong weekly seasonality + zero-revenue weekends/holidays. The forecaster's reindex-to-zero step is intentional.
- Returns are real and material; do not drop negative quantities.
- Long tail of SKUs; product-level forecasting would need very different data assumptions than the current single-series approach.
- 99th-percentile anomaly thresholds catch the extremes but will accept legitimate B2B bulk orders — this is acknowledged in the model card.